In [34]:
!pip install pyarrow

In [35]:
!pip install fastparquet
!pip install --upgrade pyarrow

# LEVEL MACRO

In [36]:
import pandas as pd

df0_bus = pd.read_parquet('../Data/raw_data/business_counts/Business_counts_IS8_LADs.parquet', engine='fastparquet')
df0_em  = pd.read_parquet('../Data/raw_data/employee_counts/Employee_counts_IS8_LADs.parquet',  engine='fastparquet')

keys = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 'IS8_SECTOR', 'FRONTIER_SECTOR']

df0_bus = df0_bus.dropna(subset=['IS8_SECTOR'])
df0_em  = df0_em.dropna(subset=['IS8_SECTOR'])

# I needed to group by sector because the database was too heavy.
def classify_frontier(row):
    if row['IS8_SECTOR'] == 'Digital and Technology':
        return 'Digital and Technology'
    elif row['IS8_SECTOR'] == 'Defence sector':
        return 'Defence Sector'
    else:
        return row['IS8_SECTOR']

df0_bus['FRONTIER_SECTOR'] = df0_bus.apply(classify_frontier, axis=1)
df0_em['FRONTIER_SECTOR']  = df0_em.apply(classify_frontier, axis=1)

df0_bus_grouped = df0_bus.groupby(keys, as_index=False)['OBS_VALUE'].sum()
df0_bus_grouped = df0_bus_grouped.rename(columns={'OBS_VALUE': 'OBS_Value_Business'})

df0_em_grouped = df0_em.groupby(keys, as_index=False)['OBS_VALUE'].sum()
df0_em_grouped = df0_em_grouped.rename(columns={'OBS_VALUE': 'OBS_Value_Employment'})

df_merged = df0_bus_grouped.merge(df0_em_grouped, on=keys, how='left')

for col in keys:
    df_merged[col] = df_merged[col].astype('category')

df_merged['OBS_Value_Business']   = df_merged['OBS_Value_Business'].astype('float32')
df_merged['OBS_Value_Employment'] = df_merged['OBS_Value_Employment'].astype('float32')

# LEVEL MICRO

In [37]:
import pandas as pd

df1_bus = pd.read_parquet('../Data/raw_data/business_counts/Business_counts_IS8_MSOAs.parquet', engine='fastparquet')
df1_em  = pd.read_parquet('../Data/raw_data/employee_counts/Employee_counts_IS8_MSOAs.parquet',  engine='fastparquet')

keys = ['YEAR', 'GEOGRAPHY_CODE', 'GEOGRAPHY_NAME', 'IS8_SECTOR', 'FRONTIER_SECTOR']

df1_bus = df1_bus.dropna(subset=['IS8_SECTOR'])
df1_em  = df1_em.dropna(subset=['IS8_SECTOR'])

# I needed to groupby per Sector because the database was too heavy.
df1_bus_grouped = df1_bus.groupby(keys, as_index=False)['OBS_VALUE'].sum()
df1_bus_grouped = df1_bus_grouped.rename(columns={'OBS_VALUE': 'OBS_Value_Business'})

df1_em_grouped = df1_em.groupby(keys, as_index=False)['OBS_VALUE'].sum()
df1_em_grouped = df1_em_grouped.rename(columns={'OBS_VALUE': 'OBS_Value_Employment'})

df_merged_msoa = df1_bus_grouped.merge(df1_em_grouped, on=keys, how='left')

for col in keys:
    df_merged_msoa[col] = df_merged_msoa[col].astype('category')

df_merged_msoa['OBS_Value_Business']   = df_merged_msoa['OBS_Value_Business'].astype('float32')
df_merged_msoa['OBS_Value_Employment'] = df_merged_msoa['OBS_Value_Employment'].astype('float32')

df_merged_msoa.head()

# MERGE WITH DEMOGRAPHIC DATA

In [38]:
import pandas as pd

file = '../Data/raw_data/local_indicators/ONS_local_indicators_package.xlsx'

xls = pd.ExcelFile(file)
sheet_names = xls.sheet_names[4:]

dfs = []

for sheet_name in sheet_names:
    raw_df = pd.read_excel(file, sheet_name=sheet_name, header=None)

    area_row_idx = raw_df[raw_df.iloc[:, 0].astype(str).str.contains('Area Code', na=False)].index
    if len(area_row_idx) == 0:
        print(f"Skipping sheet '{sheet_name}' because no 'Area Code' found")
        continue
    header_row = area_row_idx[0]

    df = pd.read_excel(file, sheet_name=sheet_name, header=header_row)
    df = df.dropna(axis=1, how='all')

    numeric_cols = []
    for col in df.columns:
        temp = pd.to_numeric(df[col], errors='coerce')
        if temp.notna().any():
            numeric_cols.append(col)

    if not numeric_cols:
        continue

    id_cols = [c for c in df.columns if c not in numeric_cols]

    df_long = df.melt(
        id_vars=id_cols,
        value_vars=numeric_cols,
        var_name='Metric',
        value_name='Metric_Value'
    )

    df_long['Metric_Value'] = pd.to_numeric(df_long['Metric_Value'], errors='coerce')
    df_long['Sheet'] = sheet_name
    dfs.append(df_long)

df_all = pd.concat(dfs, ignore_index=True)

code_cols = [c for c in df_all.columns if 'Code' in c or 'Area' in c]
for col in code_cols:
    df_all[col] = df_all[col].astype('category')

print(df_all.info())
df_all.head()

In [39]:
df_all['Area Code']            = df_all['Area Code'].astype(str).str.strip()
df_merged['GEOGRAPHY_CODE']    = df_merged['GEOGRAPHY_CODE'].astype(str).str.strip()

matched = df_all['Area Code'].isin(df_merged['GEOGRAPHY_CODE'])
print(f'Matched: {matched.sum()} / {len(df_all)} rows')

In [40]:
df_grouped = (
    df_all
    .groupby(['Area Code', 'Metric'], as_index=False)
    .agg({'Metric_Value': 'mean'})
)

df_grouped.head()

In [41]:
import pandas as pd

df_grouped['Area Code']         = df_grouped['Area Code'].astype(str).str.strip()
df_merged['GEOGRAPHY_CODE']     = df_merged['GEOGRAPHY_CODE'].astype(str).str.strip()

df_merged_2022 = df_merged[df_merged['YEAR'] == 2022].copy()

df_metrics_wide = df_grouped.pivot_table(
    index='Area Code',
    columns='Metric',
    values='Metric_Value',
    aggfunc='first'
).reset_index()
df_metrics_wide.columns.name = None

df_merged_2022_final = df_merged_2022.merge(
    df_metrics_wide,
    left_on='GEOGRAPHY_CODE',
    right_on='Area Code',
    how='left'
)
df_merged_2022_final = df_merged_2022_final.drop(columns=['Area Code'])

df_final = pd.concat([
    df_merged[df_merged['YEAR'] != 2022],
    df_merged_2022_final
], ignore_index=True)

print(df_final.info())
df_final.head()

In [42]:
df_final.to_csv('../Data/cleaning_data/IS8_Business_Employment_Demography.csv', index=False)
df_merged_msoa.to_csv('../Data/cleaning_data/MSOA_Business_Employment.csv', index=False)

# Remember the demographic data is from 2022 only. So, the rest is missing value.

# COMBINED REGION MAPPING

In [ ]:
import pandas as pd

region_map = {
    # 1. South Yorkshire Mayoral Combined Authority
    'Barnsley': 'South Yorkshire MCA',
    'Doncaster': 'South Yorkshire MCA',
    'Rotherham': 'South Yorkshire MCA',
    'Sheffield': 'South Yorkshire MCA',
    # 2. West Midlands Combined Authority
    'Birmingham': 'West Midlands CA',
    'Coventry': 'West Midlands CA',
    'Dudley': 'West Midlands CA',
    'Sandwell': 'West Midlands CA',
    'Solihull': 'West Midlands CA',
    'Walsall': 'West Midlands CA',
    'Wolverhampton': 'West Midlands CA',
    # 3. West Yorkshire Combined Authority
    'Bradford': 'West Yorkshire CA',
    'Calderdale': 'West Yorkshire CA',
    'Kirklees': 'West Yorkshire CA',
    'Leeds': 'West Yorkshire CA',
    'Wakefield': 'West Yorkshire CA',
    # 4. Greater Manchester Combined Authority
    'Bolton': 'Greater Manchester CA',
    'Bury': 'Greater Manchester CA',
    'Manchester': 'Greater Manchester CA',
    'Oldham': 'Greater Manchester CA',
    'Rochdale': 'Greater Manchester CA',
    'Salford': 'Greater Manchester CA',
    'Stockport': 'Greater Manchester CA',
    'Tameside': 'Greater Manchester CA',
    'Trafford': 'Greater Manchester CA',
    'Wigan': 'Greater Manchester CA',
    # 5. West of England Combined Authority
    'Bristol, City of': 'West of England CA',
    'Bath and North East Somerset': 'West of England CA',
    'South Gloucestershire': 'West of England CA',
    # 6. Glasgow City Region
    'Glasgow City': 'Glasgow City Region',
    'East Dunbartonshire': 'Glasgow City Region',
    'East Renfrewshire': 'Glasgow City Region',
    'Inverclyde': 'Glasgow City Region',
    'North Lanarkshire': 'Glasgow City Region',
    'Renfrewshire': 'Glasgow City Region',
    'South Lanarkshire': 'Glasgow City Region',
    'West Dunbartonshire': 'Glasgow City Region',
    # 7. Cardiff-Newport Metropolitan Area
    'Cardiff': 'Cardiff-Newport Metro',
    'Newport': 'Cardiff-Newport Metro',
    'Caerphilly': 'Cardiff-Newport Metro',
    'Blaenau Gwent': 'Cardiff-Newport Metro',
    'Torfaen': 'Cardiff-Newport Metro',
    'Monmouthshire': 'Cardiff-Newport Metro',
    'Rhondda Cynon Taf': 'Cardiff-Newport Metro',
    'Merthyr Tydfil': 'Cardiff-Newport Metro',
    'Vale of Glamorgan': 'Cardiff-Newport Metro',
    'Bridgend': 'Cardiff-Newport Metro'
}

df_final['combined_region'] = df_final['GEOGRAPHY_NAME'].map(region_map)
df_final['combined_region'] = df_final['combined_region'].fillna('Other / Not in selected regions')

# ANALYSIS

In [ ]:
# Total businesses by region and sector
crosstab = (
    df_final
    .groupby(['combined_region', 'IS8_SECTOR'])['OBS_Value_Business']
    .sum()
    .sort_values(ascending=False)
)

crosstab

In [ ]:
# Pivot to put years as columns
df_wide = df_final.pivot_table(
    index=['combined_region', 'FRONTIER_SECTOR'],
    columns='YEAR',
    values='OBS_Value_Business',
    aggfunc='sum'
).reset_index()

# Compute growth rate
df_wide['growth_rate'] = (df_wide[2025] - df_wide[2022]) / df_wide[2022]

df_wide